In [43]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
#from NeuralMF import NeuralMF
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import pytorch_lightning as pl
import optuna
from sklearn.preprocessing import StandardScaler
from optuna.integration import PyTorchLightningPruningCallback
import gc  # 가비지 컬렉션
from sklearn.metrics import mean_squared_error, mean_absolute_error


In [3]:
!pip install pytorch_lightning
!pip install optuna
!pip install optuna-integration[pytorch_lightning]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.0/823.0 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.9/960.9 kB 58.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlin

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# movie_ratings = pd.read_csv('./dataset/ratings.csv')
# movie_ratings_small = pd.read_csv('./dataset/ratings_small.csv')
# movies_metadata = pd.read_csv('./dataset/movies_metadata.csv')

movie_ratings = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movie_dataset/ratings.csv')
movie_ratings_small = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movie_dataset/ratings_small.csv')
movies_metadata = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movie_dataset/movies_metadata.csv')


<ipython-input-7-80c1eb9504cb>:7: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_metadata = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movie_dataset/movies_metadata.csv')


In [8]:
movie_ratings.isnull().sum()

,0
userId,0
movieId,0
rating,0
timestamp,0


In [9]:
movies_metadata.drop_duplicates(subset='id',keep='first', inplace=True)

In [10]:
movies_metadata = movies_metadata[movies_metadata['id'].str.isdigit()]
movies_metadata['id'] = movies_metadata['id'].astype('int64')

<ipython-input-10-7420a33fdfa0>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_metadata['id'] = movies_metadata['id'].astype('int64')


In [11]:
movies_metadata = movies_metadata.merge(movie_ratings_small, left_on='id', right_on='movieId', how='left')

In [12]:
movies_metadata.dropna(subset='userId', inplace= True)
movies_metadata.drop(columns=['movieId'], inplace=True)

In [13]:
movies_metadata.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count', 'userId', 'rating', 'timestamp'],
      dtype='object')

In [14]:
movies_metadata['genres']

,genres
5,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam..."
6,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam..."
7,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam..."
8,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam..."
9,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam..."
...,...
87527,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ..."
87528,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ..."
87529,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ..."
87532,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '..."


In [15]:
import json
def extract_genre_names(metadata):
    # 각 리스트에서 'name' 키 값을 추출
    corrected_json_string = metadata.replace("'", '"')
    json_data = json.loads(corrected_json_string)
    a  = [genre['name'] for genre in json_data if 'name' in genre]
    return a

movies_metadata['genres'] = movies_metadata['genres'].apply(lambda x: extract_genre_names(x))


In [16]:
users_stats = movies_metadata.groupby('userId')['rating'].agg(['mean','std','count']).reset_index()
users_stats.columns = ['userId','user_mean_rating','user_rating_std','user_review_count']
movies_metadata = movies_metadata.merge(users_stats, on='userId', how='left')

In [17]:
movies_metadata['release_year'] = pd.to_datetime(movies_metadata['release_date']).dt.year

movie_stats = movies_metadata.groupby('id')['rating'].agg(['mean', 'count']).reset_index()
movie_stats.columns = ['id', 'movie_mean_rating', 'movie_review_count']

movies_metadata = movies_metadata.merge(movie_stats, on='id', how='left')


In [18]:
movies_metadata.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count', 'userId', 'rating', 'timestamp',
       'user_mean_rating', 'user_rating_std', 'user_review_count',
       'release_year', 'movie_mean_rating', 'movie_review_count'],
      dtype='object')

In [19]:
unique_genres = sorted(set(genre for genres in movies_metadata['genres'] for genre in genres))
genre_to_idx = {genre: idx for idx, genre in enumerate(unique_genres)}


# 장르를 숫자로 변환
movies_metadata['genre_ids'] = movies_metadata['genres'].apply(lambda x: [genre_to_idx[genre] for genre in x])

In [20]:
genre_ratings = movies_metadata.explode('genre_ids').groupby(['userId','genre_ids'])['rating'].mean().reset_index()
genre_ratings.columns = ['userId','genre_ids', 'user_preference']


In [21]:
movies_metadata_exploded = movies_metadata.explode('genre_ids')
movies_metadata_exploded.fillna({'genre_ids': 0}, inplace= True)



<ipython-input-21-2d472db91df7>:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  movies_metadata_exploded.fillna({'genre_ids': 0}, inplace= True)


In [22]:
movies_metadata_exploded = movies_metadata_exploded.merge(genre_ratings, on=['userId','genre_ids'], how='left')

In [23]:
movies_metadata_exploded.isnull().sum()

,0
adult,0
belongs_to_collection,80502
budget,0
genres,0
homepage,81995
id,0
imdb_id,0
original_language,0
original_title,0
overview,150


In [24]:
movies_metadata_exploded[[ 'user_mean_rating', 'user_rating_std']].isnull().sum()


,0
user_mean_rating,0
user_rating_std,0


In [25]:
numeric_features_cols = ['user_mean_rating', 'user_rating_std', 'user_review_count',
       'release_year', 'movie_mean_rating', 'movie_review_count', 'genre_ids',
       'user_preference']
movies_metadata_exploded.fillna({'release_year':0}, inplace=True)
movies_metadata_exploded.fillna({'user_preference':0}, inplace=True)
movies_metadata_exploded.fillna({'user_rating_std':0}, inplace= True)

In [26]:
numeric_features_cols = ['user_mean_rating', 'user_rating_std', 'user_review_count',
       'release_year', 'movie_mean_rating', 'movie_review_count', 'genre_ids',
       'user_preference']

scaler = StandardScaler()
movies_metadata_exploded[numeric_features_cols] = scaler.fit_transform(movies_metadata_exploded[numeric_features_cols])

In [27]:
target = ['user_mean_rating', 'user_rating_std', 'user_review_count',
       'release_year', 'movie_mean_rating', 'movie_review_count', 'genre_ids',
       'user_preference','id','userId','rating']
movies_metadata_exploded[target]

,user_mean_rating,user_rating_std,user_review_count,release_year,movie_mean_rating,movie_review_count,genre_ids,user_preference,id,userId,rating
0,0.207586,-0.354892,0.669992,0.217014,0.049479,-0.826605,-1.405291,0.245843,949,23.0,3.5
1,0.207586,-0.354892,0.669992,0.217014,0.049479,-0.826605,-0.686947,0.574224,949,23.0,3.5
2,0.207586,-0.354892,0.669992,0.217014,0.049479,-0.826605,-0.327775,0.149901,949,23.0,3.5
3,0.207586,-0.354892,0.669992,0.217014,0.049479,-0.826605,1.647671,0.354781,949,23.0,3.5
4,0.949350,-0.427886,0.654369,0.217014,0.049479,-0.826605,-1.405291,0.721596,949,102.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...
110073,0.733337,-1.364769,-0.840205,0.659329,0.817022,-1.027519,-0.866533,0.938690,98604,352.0,4.0
110074,0.733337,-1.364769,-0.840205,0.659329,0.817022,-1.027519,1.108913,0.516978,98604,352.0,4.0
110075,0.320610,0.231559,-0.199674,-2.254746,2.706358,-1.027519,0.031397,1.165766,49280,187.0,5.0
110076,0.320610,0.231559,-0.199674,-2.254746,2.706358,-1.027519,-1.405291,0.268912,49280,187.0,5.0


In [29]:
movies_metadata_exploded[target].isnull().sum()

,0
user_mean_rating,0
user_rating_std,0
user_review_count,0
release_year,0
movie_mean_rating,0
movie_review_count,0
genre_ids,0
user_preference,0
id,0
userId,0


In [32]:
# Neural MF Model
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl

class NeuralMF(pl.LightningModule):
    def __init__(self, num_users, num_items, num_numeric_features, latent_dim, dropout, learning_rate=0.001):
        super(NeuralMF, self).__init__()

        # Embeddings
        self.user_embedding = nn.Embedding(num_users, latent_dim, max_norm=1.0)
        self.item_embedding = nn.Embedding(num_items, latent_dim, max_norm=1.0)
        self.numeric_fc = nn.Linear(num_numeric_features, latent_dim)
        # Neural network layers
        self.gmf_fc = nn.Linear(latent_dim, latent_dim)
        # MLP (Multi-Layer Perceptron)
        self.mlp_fc1 = nn.Linear(latent_dim * 3, 128)
        self.mlp_fc2 = nn.Linear(128, 64)
        self.final_fc = nn.Linear(latent_dim + 64, 1)  # GMF(latent_dim) + MLP(64)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()
        self.learning_rate = learning_rate
        self.criterion = nn.MSELoss()

    def forward(self, user, item, numeric_features):
        user_embed = self.user_embedding(user)
        item_embed = self.item_embedding(item)
        numeric_emb = self.numeric_fc(numeric_features)
        gmf = user_embed * item_embed  # 원소별 곱
        gmf = self.gmf_fc(gmf)  # (batch_size, latent_dim)
        # Concatenate user and item embeddings
        mlp = torch.cat([user_embed, item_embed, numeric_emb], dim=1)
        mlp = self.activation(self.mlp_fc1(mlp))
        mlp = self.dropout(mlp)
        mlp = self.activation(self.mlp_fc2(mlp))  # (batch_size, 64)

        x = torch.cat([gmf, mlp], dim=1)  # (batch_size, latent_dim + 64)
        x = self.final_fc(x)  # (batch_size, 1)
        return (torch.sigmoid(x) * 5).squeeze()

    def training_step(self, batch, batch_idx):
        user_ids, item_ids, numeric_features, ratings = batch
        predicted_ratings = self.forward(user_ids, item_ids, numeric_features)
        train_loss = self.criterion(predicted_ratings, ratings)
        validation_loss = self.trainer.callback_metrics.get("validation_loss", torch.tensor(float('inf'))).item()

        # 🔥 Train과 Validation Loss 차이를 계산
        loss_gap = validation_loss - train_loss.item()

        self.log("loss_gap", loss_gap, prog_bar=True, on_epoch=True, on_step=True)
        # self.log("train_loss", loss, prog_bar=True, on_epoch=True,  on_step = True)  # 🔥 손실 로깅 추가
        return train_loss

    def validation_step(self, batch, batch_idx):
        user_ids, item_ids, numeric_features, ratings = batch
        predicted_ratings = self.forward(user_ids, item_ids, numeric_features)
        loss = self.criterion(predicted_ratings, ratings)
        self.log("validation_loss", loss, prog_bar=True, on_epoch=True, on_step = True)  # 🔥 손실 로깅 추가
        return loss

    def predict_step(self, batch, batch_idx):
        """Lightning에서 `trainer.predict()`를 호출할 때 사용"""
        user, item, numeric_features, ratings = batch  # 배치에서 올바른 입력 추출
        prediction = self.forward(user, item, numeric_features)
        return prediction, ratings

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.learning_rate,  weight_decay=1e-3)

In [36]:
from optuna.integration import PyTorchLightningPruningCallback
data = movies_metadata_exploded[target]
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)
small_train_data = train_data.sample(frac=0.2, random_state=42)
val_data, test_data = train_test_split(val_data, test_size=0.5, random_state=42)  # 검증 & 테스트 분리

def objective(trial):
    latent_dim = trial.suggest_int("latent_dim", 8, 128)  # 8~64 사이 정수
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64,128])  # 16, 32, 64 중 선택
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True)  # log=True 사용
    epochs = trial.suggest_int("epochs", 10, 50)  # 10~50 사이 정수
    num_workers = trial.suggest_int("num_workers", 0, 4)  # 🔥 num_workers도 최적화 가능!
    dropout = trial.suggest_float("dropout", 0.0, 0.7)  # 드롭아웃 비율


    def prepare(data, is_train):
      data['user_idx'] = data['userId'].astype('category').cat.codes
      data['item_idx'] = data['id'].astype('category').cat.codes
      user_ids = torch.tensor(data['user_idx'].values, dtype=torch.long)
      item_ids = torch.tensor(data['item_idx'].values, dtype=torch.long)
      ratings = torch.tensor(data['rating'].values, dtype=torch.float32)
      num_users = data['user_idx'].nunique()
      num_items = data['item_idx'].nunique()
      numeric_features = torch.tensor(data[numeric_features_cols].values, dtype=torch.float32)
      dataset = TensorDataset(user_ids, item_ids, numeric_features, ratings)
      loader = DataLoader(dataset, batch_size=batch_size, shuffle= is_train, num_workers= num_workers)
      return loader, num_users, num_items, numeric_features.shape[1]

    train_loader, num_users, num_items, num_numeric = prepare(small_train_data, True)
    val_loader, _, _, _ = prepare(val_data, False)

    model = NeuralMF(num_users, num_items, num_numeric,  latent_dim, dropout, learning_rate)
    pruning_callback = PyTorchLightningPruningCallback(trial, monitor="loss_gap")
    trainer = pl.Trainer(
        max_epochs=epochs,
        enable_checkpointing=False, # 체크포인트 저장 비활성화
        enable_progress_bar=False, # 진행 바 비활성화
        logger=False # 로그 저장 비활성화
    )
    trainer.callbacks.append(pruning_callback)  # 🔥 여기서 직접 추가
    trainer.fit(model, train_loader, val_loader)
    train_loss = trainer.callback_metrics.get("train_loss", torch.tensor(float('inf'))).item()
    validation_loss = trainer.callback_metrics.get("validation_loss", torch.tensor(float('inf'))).item()
    loss_gap = train_loss - validation_loss  # 🔥 loss_gap을 최종 반환
    # del train_loader
    # del val_loader
    gc.collect()  # 🔥 가비지 컬렉션 실행
    torch.cuda.empty_cache()  # 🔥 GPU 캐시 정리 (GPU 사용 시)
    return loss_gap

# ✅ Optuna 실행
study = optuna.create_study(
        study_name="newMF_study",
        storage="sqlite:///optuna_results.db")  # 최소화 목표: 손실을 최소화하려고 함
study.optimize(objective, n_trials=20)  # n_trials는 실험 횟수

# ✅ 최적 하이퍼파라미터 출력
print("Best Hyperparameters:", study.best_params)

[I 2025-03-25 04:59:30,151] A new study created in RDB with name: newMF_study
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name           | Type      | Params | Mode 
-----------------------------------------------------
0 | user_embedding | Embedding | 74.5 K | train
1 | item_embedding | Embedding | 221 K  | train
2 | numeric_fc     | Linear    | 1.0 K  | train
3 | gmf_fc         | Linear    | 12.7 K | train
4 | mlp_fc1        | Linear    | 43.1 K | train
5 | mlp_fc2        | Linear    | 8.3 K  | train
6 | final_fc       | L

Best Hyperparameters: {'latent_dim': 112, 'batch_size': 128, 'learning_rate': 0.00011357020401617381, 'epochs': 42, 'num_workers': 0, 'dropout': 0.4060787393086205}


In [37]:
print("Best Hyperparameters:", study.best_params)

Best Hyperparameters: {'latent_dim': 112, 'batch_size': 128, 'learning_rate': 0.00011357020401617381, 'epochs': 42, 'num_workers': 0, 'dropout': 0.4060787393086205}


In [38]:
print(f"Number of trials: {len(study.trials)}")
print(f"Completed trials: {[t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]}")


Number of trials: 20
Completed trials: [FrozenTrial(number=0, state=1, values=[inf], datetime_start=datetime.datetime(2025, 3, 25, 4, 59, 30, 158936), datetime_complete=datetime.datetime(2025, 3, 25, 5, 0, 17, 674872), params={'latent_dim': 112, 'batch_size': 128, 'learning_rate': 0.00011357020401617381, 'epochs': 42, 'num_workers': 0, 'dropout': 0.4060787393086205}, user_attrs={}, system_attrs={}, intermediate_values={0: inf, 1: -0.0009090304374694824, 2: 0.11220228672027588, 3: 0.0691525936126709, 4: -0.2426459789276123, 5: 0.04700970649719238, 6: 0.02738773822784424, 7: 0.07087302207946777, 8: -0.04460674524307251, 9: -0.27453309297561646, 10: 0.0019083023071289062, 11: 0.13027268648147583, 12: 0.12325209379196167, 13: 0.20788320899009705, 14: -0.23681306838989258, 15: -0.08625483512878418, 16: 0.19841599464416504, 17: -0.031978487968444824, 18: -0.020531535148620605, 19: -0.22348660230636597, 20: 0.08323246240615845, 21: 0.1727910041809082, 22: 0.15799438953399658, 23: 0.1465094685

In [54]:
#Best Hyperparameters: {'latent_dim': 43, 'batch_size': 32, 'learning_rate': 0.000532381383106112, 'epochs': 11, 'num_workers': 1}

#best_params = study.best_params

data = movies_metadata_exploded[target]
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)
small_train_data = train_data.sample(frac=0.2, random_state=42)
val_data, test_data = train_test_split(val_data, test_size=0.5, random_state=42)  # 검증 & 테스트 분리

#best_params =  {'latent_dim': 24, 'batch_size': 16, 'learning_rate': 0.00010468856330302876, 'epochs': 24, 'num_workers': 3,  'dropout': 0.47353989106173777}
best_params = {
    'latent_dim': 112,
    'batch_size': 128,
    'learning_rate': 0.00011357020401617381,
    'epochs': 42,
    'num_workers': 0,
    'dropout': 0.4060787393086205
}
latent_dim = best_params['latent_dim']
batch_size = best_params['batch_size']
learning_rate = best_params['learning_rate']
epochs = best_params['epochs']
num_workers = best_params['num_workers']
dropout = best_params['dropout']


def prepare(data, is_train):
    data['user_idx'] = data['userId'].astype('category').cat.codes
    data['item_idx'] = data['id'].astype('category').cat.codes
    user_ids = torch.tensor(data['user_idx'].values, dtype=torch.long)
    item_ids = torch.tensor(data['item_idx'].values, dtype=torch.long)
    ratings = torch.tensor(data['rating'].values, dtype=torch.float32)
    num_users = data['user_idx'].nunique()
    num_items = data['item_idx'].nunique()
    numeric_features = torch.tensor(data[numeric_features_cols].values, dtype=torch.float32)
    dataset = TensorDataset(user_ids, item_ids, numeric_features, ratings)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle= is_train, num_workers= 2)
    return loader, num_users, num_items, numeric_features.shape[1]

train_loader, num_users, num_items, num_numeric = prepare(train_data, True)
val_loader, _, _, _ = prepare(val_data, False)

model = NeuralMF(num_users, num_items, num_numeric, latent_dim, dropout, learning_rate)
trainer = pl.Trainer(
    max_epochs=epochs,
    enable_checkpointing=False, # 체크포인트 저장 비활성화
    enable_progress_bar=False, # 진행 바 비활성화
    logger=False # 로그 저장 비활성화
)
trainer.fit(model, train_loader, val_loader)

# 모델 학습이 끝난 후 불필요한 변수 제거
#del train_loader, val_loader
torch.cuda.empty_cache()  # GPU 메모리 해제
gc.collect()  # Python 가비지 컬렉션 실행



INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name           | Type      | Params | Mode 
-----------------------------------------------------
0 | user_embedding | Embedding | 75.2 K | train
1 | item_embedding | Embedding | 311 K  | train
2 | numeric_fc     | Linear    | 1.0 K  | train
3 | gmf_fc         | Linear    | 12.7 K | train
4 | mlp_fc1        | Linear    | 43.1 K | train
5 | mlp_fc2        | Linear    | 8.3 K  | train
6 | final_fc       | Linear    | 177    | train
7 | dropout        | Dropout   | 0      | train
8 | 

775

In [55]:
test_loader, _, _, _ = prepare(test_data, False)
predictions = trainer.predict(model, test_loader)


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


In [129]:
for user_ids, item_ids, numeric_features, ratings in test_loader:
  print(user_ids.shape)
  print(item_ids.shape)
  print(numeric_features.shape)
  print(ratings.shape)
  break

torch.Size([128])
torch.Size([128])
torch.Size([128, 8])
torch.Size([128])


In [85]:
num_users,  num_items

(671, 2779)

In [41]:
predicted_ratings = []
actual_ratings = []
for pred, rating in predictions:
    predicted_ratings.append(pred)
    actual_ratings.append(rating)

In [44]:
# ✅ MSE 계산 (sklearn)
mse = mean_squared_error(actual_ratings, predicted_ratings)
print(f"Test MSE: {mse}")

Test MSE: 0.8107448788708513


In [45]:
mae = mean_absolute_error(actual_ratings, predicted_ratings)
print(f"Test MAE: {mae}")

Test MAE: 0.6894948442769777


In [46]:
#RMSE 계산
rmse = np.sqrt(mse)
print(f"Test RMSE: {rmse}")

Test RMSE: 0.9004137265006855


In [76]:
torch.save(model.state_dict(), "best_model.pth")

In [116]:
# 유저 1의 user_idx 찾기
user_id = 2
item_title = "장화, 홍련"
item_id = int(movies_metadata_exploded[movies_metadata_exploded['original_title'] == item_title]['id'].iloc[0])

user_idx = train_data[train_data['userId'] == user_id]['user_idx'].unique()[0]
item_idx = train_data[train_data['id'] == item_id]['item_idx'].unique()[0]

print(f"유저 {user_id} user_index: {user_idx}, {item_id} item_index: {item_idx}")

numeric_item_features = train_data[train_data['item_idx'] == item_idx][numeric_features_cols]

numeric_item_features.mean().values


유저 2 user_index: 1, 4552 item_index: 1596


array([ 0.14000068,  0.22755207, -0.17710736,  0.42516245,  1.0531892 ,
       -0.9873358 ,  0.53024706, -0.20964886])

In [ ]:
user_ids = torch.tensor([user_idx], dtype=torch.long)  # 예: 사용자 아이디
item_ids = torch.tensor([item_idx], dtype=torch.long)  # 예: 아이템 아이디
numeric_features = torch.tensor(numeric_item_features.mean().values,  dtype=torch.float32)  # 예: 숫자형 특성 (예: 나이, 성별, 등)
item_features = numeric_features.reshape(1, -1)
# 모델을 통해 예측 수행
predicted_rating = model(user_ids, item_ids, item_features)
# 예측된 평점 출력
print(predicted_rating.item())

4.16526460647583
